# Analytics Layer - Business Intelligence Views

Creating five SQL views in the **workspace.analytics** schema for business intelligence and dashboarding purposes.

Source: workspace.silver.final_sales_complete

In [0]:
%sql
-- Create the analytics schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS workspace.analytics
COMMENT 'Analytics layer containing business intelligence views for dashboarding';

In [0]:
%sql
-- =====================================================
-- View 1: Product Performance
-- =====================================================
-- Purpose: Analyze product performance by sales, revenue, cost, and profitability
-- Grain: One row per product
-- Source: workspace.silver.final_sales_complete
-- =====================================================

CREATE OR REPLACE VIEW workspace.analytics.vw_product_performance AS
SELECT
  -- Product identifiers and attributes
  ProductKey AS product_key,
  ProductName AS product_name,
  ProductCategory AS category,
  ProductSubcategory AS subcategory,
  ProductColor AS color,
  
  -- Order and quantity metrics
  COUNT(DISTINCT OrderNumber) AS total_orders,
  SUM(OrderQuantity) AS total_quantity_sold,
  
  -- Financial metrics
  SUM(Revenue) AS total_revenue,
  SUM(TotalCost) AS total_cost,
  SUM(Profit) AS total_profit,
  
  -- Average pricing and costs
  AVG(UnitPrice) AS avg_unit_price,
  AVG(UnitCost) AS avg_unit_cost,
  
  -- Profit margin calculation (safe division)
  CASE 
    WHEN SUM(Revenue) > 0 THEN (SUM(Profit) / SUM(Revenue)) * 100
    ELSE 0
  END AS avg_profit_margin_pct,
  
  -- Customer engagement metrics
  COUNT(DISTINCT CustomerKey) AS distinct_customers,
  COUNT(DISTINCT OrderNumber) AS distinct_orders
  
FROM workspace.silver.final_sales_complete
WHERE IsValidRecord = true  -- Only include valid records
GROUP BY 
  ProductKey,
  ProductName,
  ProductCategory,
  ProductSubcategory,
  ProductColor
ORDER BY total_revenue DESC;

## Summary - Analytics Views Created

Successfully created **five SQL views** in the `workspace.analytics` schema:

### 1. **vw_product_performance**
* **Grain:** One row per product
* **Purpose:** Analyze product performance by sales, revenue, cost, and profitability
* **Key Metrics:** Total orders, quantity sold, revenue, cost, profit, profit margins, average pricing, customer engagement
* **Use Cases:** Identify top-selling products, most profitable products, pricing analysis, product portfolio optimization

### 2. **vw_customer_behavior**
* **Grain:** One row per customer
* **Purpose:** Understand customer purchasing behavior, value, and engagement patterns
* **Key Metrics:** Order frequency, purchase volume, revenue contribution, average order value, product diversity, customer lifetime metrics
* **Use Cases:** Customer segmentation, lifetime value analysis, loyalty programs, churn prediction

### 3. **vw_daily_analytics**
* **Grain:** One row per order date
* **Purpose:** Track daily sales performance and identify time-based trends
* **Key Metrics:** Daily orders, revenue, profit, margins, customer and product diversity per day
* **Use Cases:** Time-series analysis, seasonal patterns, weekly trends, sales forecasting, performance monitoring

### 4. **vw_region_analytics**
* **Grain:** One row per region-country combination
* **Purpose:** Compare sales performance across geographical regions
* **Key Metrics:** Regional sales, revenue, profitability, average order value, market penetration (customers, products, cities)
* **Use Cases:** Market analysis, regional performance comparison, expansion planning, territory management

### 5. **vw_customer_product_breakdown**
* **Grain:** One row per customer-product combination
* **Purpose:** Analyze customer-product relationships and purchase patterns
* **Key Metrics:** Purchase frequency, quantity, revenue by customer-product pair, pricing, repeat purchase behavior
* **Use Cases:** Cross-sell/upsell opportunities, product recommendations, customer preference analysis, basket analysis

---

**Data Quality:** All views filter on `IsValidRecord = true` to ensure data quality.

**Safe Calculations:** All division operations include zero-check logic to prevent errors.

**Ready for BI:** These views are optimized for Power BI, Tableau, and other business intelligence tools.

In [0]:
%sql
-- =====================================================
-- View 2: Customer Behavior
-- =====================================================
-- Purpose: Understand customer purchasing behavior, value, and engagement
-- Grain: One row per customer
-- Source: workspace.silver.final_sales_complete
-- =====================================================

CREATE OR REPLACE VIEW workspace.analytics.vw_customer_behavior AS
SELECT
  -- Customer identifiers and attributes
  CustomerKey AS customer_key,
  CustomerName AS customer_name,
  CustomerGender AS gender,
  CustomerMaritalStatus AS marital_status,
  
  -- Order and quantity metrics
  COUNT(DISTINCT OrderNumber) AS total_orders,
  SUM(OrderQuantity) AS total_quantity_purchased,
  
  -- Financial metrics
  SUM(Revenue) AS total_revenue,
  SUM(TotalCost) AS total_cost,
  SUM(Profit) AS total_profit,
  
  -- Average order metrics (safe division)
  CASE 
    WHEN COUNT(DISTINCT OrderNumber) > 0 THEN SUM(Revenue) / COUNT(DISTINCT OrderNumber)
    ELSE 0
  END AS avg_order_value,
  
  CASE 
    WHEN COUNT(DISTINCT OrderNumber) > 0 THEN SUM(OrderQuantity) / COUNT(DISTINCT OrderNumber)
    ELSE 0
  END AS avg_quantity_per_order,
  
  -- Profit margin calculation (safe division)
  CASE 
    WHEN SUM(Revenue) > 0 THEN (SUM(Profit) / SUM(Revenue)) * 100
    ELSE 0
  END AS avg_profit_margin_pct,
  
  -- Product diversity metrics
  COUNT(DISTINCT ProductKey) AS distinct_products_purchased,
  COUNT(DISTINCT ProductCategory) AS distinct_categories_purchased,
  
  -- Customer lifecycle dates
  MIN(OrderDate) AS first_order_date,
  MAX(OrderDate) AS last_order_date,
  
  -- Customer lifetime duration in days
  DATEDIFF(MAX(OrderDate), MIN(OrderDate)) AS customer_lifetime_days
  
FROM workspace.silver.final_sales_complete
WHERE IsValidRecord = true  -- Only include valid records
GROUP BY 
  CustomerKey,
  CustomerName,
  CustomerGender,
  CustomerMaritalStatus
ORDER BY total_revenue DESC;

In [0]:
%sql
-- =====================================================
-- View 3: Daily Analytics
-- =====================================================
-- Purpose: Analyze daily sales performance and identify trends over time
-- Grain: One row per order date
-- Source: workspace.silver.final_sales_complete
-- =====================================================

CREATE OR REPLACE VIEW workspace.analytics.vw_daily_analytics AS
SELECT
  -- Date dimensions
  OrderDate AS order_date,
  Year AS year,
  Quarter AS quarter,
  Month AS month,
  MonthName AS month_name,
  WeekOfYear AS week_of_year,
  DayOfWeek AS day_of_week,
  
  -- Order and quantity metrics
  COUNT(DISTINCT OrderNumber) AS total_orders,
  SUM(OrderQuantity) AS total_quantity_sold,
  
  -- Financial metrics
  SUM(Revenue) AS total_revenue,
  SUM(TotalCost) AS total_cost,
  SUM(Profit) AS total_profit,
  
  -- Average order value (safe division)
  CASE 
    WHEN COUNT(DISTINCT OrderNumber) > 0 THEN SUM(Revenue) / COUNT(DISTINCT OrderNumber)
    ELSE 0
  END AS avg_order_value,
  
  -- Profit margin calculation (safe division)
  CASE 
    WHEN SUM(Revenue) > 0 THEN (SUM(Profit) / SUM(Revenue)) * 100
    ELSE 0
  END AS avg_profit_margin_pct,
  
  -- Customer and product diversity
  COUNT(DISTINCT CustomerKey) AS distinct_customers,
  COUNT(DISTINCT ProductKey) AS distinct_products
  
FROM workspace.silver.final_sales_complete
WHERE IsValidRecord = true  -- Only include valid records
GROUP BY 
  OrderDate,
  Year,
  Quarter,
  Month,
  MonthName,
  WeekOfYear,
  DayOfWeek
ORDER BY order_date DESC;

In [0]:
%sql
-- =====================================================
-- View 4: Region Analytics Dashboard
-- =====================================================
-- Purpose: Compare sales performance across geographical regions
-- Grain: One row per geographical region (Region + Country)
-- Source: workspace.silver.final_sales_complete
-- =====================================================

CREATE OR REPLACE VIEW workspace.analytics.vw_region_analytics AS
SELECT
  -- Geographic identifiers and hierarchy
  Region AS region,
  Country AS country,
  
  -- Order and quantity metrics
  COUNT(DISTINCT OrderNumber) AS total_orders,
  SUM(OrderQuantity) AS total_quantity_sold,
  
  -- Financial metrics
  SUM(Revenue) AS total_revenue,
  SUM(TotalCost) AS total_cost,
  SUM(Profit) AS total_profit,
  
  -- Average order value (safe division)
  CASE 
    WHEN COUNT(DISTINCT OrderNumber) > 0 THEN SUM(Revenue) / COUNT(DISTINCT OrderNumber)
    ELSE 0
  END AS avg_order_value,
  
  -- Profit margin calculation (safe division)
  CASE 
    WHEN SUM(Revenue) > 0 THEN (SUM(Profit) / SUM(Revenue)) * 100
    ELSE 0
  END AS avg_profit_margin_pct,
  
  -- Customer and product diversity
  COUNT(DISTINCT CustomerKey) AS distinct_customers,
  COUNT(DISTINCT ProductKey) AS distinct_products,
  COUNT(DISTINCT City) AS distinct_cities
  
FROM workspace.silver.final_sales_complete
WHERE IsValidRecord = true  -- Only include valid records
GROUP BY 
  Region,
  Country
ORDER BY total_revenue DESC;

In [0]:
%sql
-- =====================================================
-- View 5: Customer-Product Breakdown
-- =====================================================
-- Purpose: Analyze the relationship between customers and products they purchase
-- Grain: One row per customer-product combination
-- Source: workspace.silver.final_sales_complete
-- =====================================================

CREATE OR REPLACE VIEW workspace.analytics.vw_customer_product_breakdown AS
SELECT
  -- Customer identifiers and attributes
  CustomerKey AS customer_key,
  CustomerName AS customer_name,
  CustomerGender AS gender,
  
  -- Product identifiers and attributes
  ProductKey AS product_key,
  ProductName AS product_name,
  ProductCategory AS category,
  ProductSubcategory AS subcategory,
  
  -- Order and quantity metrics
  COUNT(DISTINCT OrderNumber) AS total_orders,
  SUM(OrderQuantity) AS total_quantity_purchased,
  
  -- Financial metrics
  SUM(Revenue) AS total_revenue,
  SUM(TotalCost) AS total_cost,
  SUM(Profit) AS total_profit,
  
  -- Average pricing
  AVG(UnitPrice) AS avg_unit_price,
  
  -- Profit margin calculation (safe division)
  CASE 
    WHEN SUM(Revenue) > 0 THEN (SUM(Profit) / SUM(Revenue)) * 100
    ELSE 0
  END AS avg_profit_margin_pct,
  
  -- Purchase timeline
  MIN(OrderDate) AS first_purchase_date,
  MAX(OrderDate) AS last_purchase_date
  
FROM workspace.silver.final_sales_complete
WHERE IsValidRecord = true  -- Only include valid records
GROUP BY 
  CustomerKey,
  CustomerName,
  CustomerGender,
  ProductKey,
  ProductName,
  ProductCategory,
  ProductSubcategory
ORDER BY total_revenue DESC;